In [ ]:
import pyproj
pyproj.datadir.set_data_dir("/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/proj/9.2.0/share/proj")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xarray as xr

from config_utils import load_config
from model import SimpleModel
from preprocessing import create_2024_preprocessor
from utils import (
    input_vars_from_config,
    ordered_dataset_for_config,
    plot_global_field,
    plot_north_pole_field,
    target_var_from_config,
)

SECONDS_PER_DAY = 3600 * 24
year, month = 2012, 9
output_dir = Path('closure_test_2')
output_dir.mkdir(exist_ok=True)

In [ ]:
def to_monthly(ds):
    year = ds.date.dt.year
    month = ds.date.dt.month
    ds = ds.assign_coords(year=("date", year.data), month=("date", month.data))
    return ds.set_index(date=("year", "month")).unstack("date")


def to_dates(ds):
    ds_stacked = ds.stack(date=("year", "month"))
    datetime_index = pd.to_datetime(
        {
            "year": ds_stacked.year.values,
            "month": ds_stacked.month.values,
            "day": 1,
        }
    )
    return ds_stacked.assign_coords(date=datetime_index).sortby("date")


def nn_pred(ds, model, preprocessor):
    year, month = ds.year.values, ds.month.values
    lon, lat = ds.longitude.values, ds.latitude.values
    target_var = target_var_from_config(config)

    ds_ordered_np = (
        ordered_dataset_for_config(ds, config)
        .to_dataarray()
        .transpose("year", "month", "latitude", "longitude", "variable")
        .to_numpy()
    )

    data_torch = torch.from_numpy(ds_ordered_np).float()
    model_outputs = model(data_torch[..., : model.input_dim]).detach()

    pred = (
        preprocessor.preprocessors[-1]
        .inverse_transform(
            xr.Dataset(
                data_vars={target_var: (("year", "month", "latitude", "longitude"), model_outputs)},
                coords={"year": year, "month": month, "longitude": lon, "latitude": lat},
            )
        )
        .to_dataarray()
        .squeeze(dim="variable", drop=True)
    )
    return pred


def dRnn(ds, anomaly, model, preprocessor, variables):
    ds_copies = []
    for var_list in variables:
        if not isinstance(var_list, list):
            var_list = [var_list]
        ds_copies.append(ds.assign({v: ds[v] + anomaly[v] for v in var_list}))

    ds_original = preprocessor.transform(ds)
    ds_perturbed = [preprocessor.transform(d) for d in ds_copies]

    pred_original = nn_pred(ds_original, model, preprocessor)
    return [nn_pred(ds_p, model, preprocessor) - pred_original for ds_p in ds_perturbed]


def global_mean(da):
    weights = np.cos(np.deg2rad(da.latitude))
    weights.name = "weights"
    return da.weighted(weights).mean(dim=["latitude", "longitude"])


def plot_map_pair(field, title, stem, label='W/m^2'):
    lon, lat = field.longitude, field.latitude
    vmax = float(np.abs(field).max().compute())
    ann = float(global_mean(field).compute())
    plot_global_field(field, lon, lat, title=title, save_path=output_dir/f'{stem}.png', vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label=label)
    plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir/f'{stem}_np.png', vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label=label)


def global_date_series(da, scale=1):
    if {'year', 'month'} <= set(da.dims):
        da = to_dates(da.copy(deep=True))
    return (global_mean(da) / scale).compute()


def plot_timeseries_comparison(series, var):
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.axhline(0, alpha=0.1)
    for source in ['nn', 'k']:
        name = f'dR_{var}_{source}'
        ax.scatter(x=series[name]['date'], y=series[name], s=1, label=name)
    ax.set_yticks([-2, 0, 2])
    ax.set_xticks(pd.date_range(start='2007', end='2017', freq='YS', inclusive='both'), np.arange(2007, 2018))
    ax.set_ylabel('W/m^2')
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / f'dR_{var}_nn_vs_kernel_timeseries.png', dpi=200)
    plt.show()


def integrate_over_pressure_levels(sp, da):
    p = da.level * 100.0
    return da.where(p <= sp, 0).sum(dim="level")

## Model And Data

In [ ]:
config = load_config("configs/model/fal/sobolev_mar_june_sep_dec_2011_2014.yaml")
checkpoint_data = torch.load(
    Path(config.train.checkpoint_dir) / "best_model.pt",
    map_location="cpu",
    weights_only=False,
)

model = SimpleModel(checkpoint_data["config"])
model.load_state_dict(checkpoint_data["model_state_dict"])
model.eval()

preprocessor = create_2024_preprocessor(
    input_vars=input_vars_from_config(config),
    target_var=target_var_from_config(config),
    ecod=config.preprocess.ecod,
)
preprocessor.load(model.config.preprocess.params_dir)

In [ ]:
data_path = Path('data/era5')
ds = xr.open_mfdataset(data_path.glob('era5_single_levels_monthly_*.nc'))
ds = ds.sel(date=slice('2007-01', '2016-12'))

ds_monthly = to_monthly(ds.copy(deep=True))
ds_monthly_means = ds_monthly.mean('year')
anomaly = ds_monthly - ds_monthly_means

## Input Anomalies

In [ ]:
for var, label in zip(['fal', 'hcc', 'mcc', 'lcc', 'tcwv', 'tsr'], [''] * 5 + ['W/m^2']):
    field = anomaly.sel(month=month, year=year)[var]
    if var == 'tsr':
        field = field / SECONDS_PER_DAY
    plot_map_pair(field, r'$\Delta$' + f'{var}, {year}-{month:02d}', f'delta_{var}', label=label)

## Neural Network Responses

In [ ]:
dR_components = dRnn(
    ds_monthly,
    anomaly,
    model,
    preprocessor,
    ['fal', ['tcc', 'hcc', 'mcc', 'lcc', 'tciw', 'tclw'], 'tcwv'],
)

dR_a_nn, dR_c_nn, dR_q_nn = dR_components

In [ ]:
for dR, label in [(dR_a_nn, 'a'), (dR_c_nn, 'c'), (dR_q_nn, 'q')]:
    field = dR.sel(month=month, year=year) / SECONDS_PER_DAY
    plot_map_pair(field, r'$\Delta R$' + f'{label}, {year}-{month:02d}', f'dR_{label}')

dR_sum_nn = dR_a_nn + dR_c_nn + dR_q_nn
dR_nn_residual = anomaly.tsr.interp(latitude=dR_sum_nn.latitude, longitude=dR_sum_nn.longitude) - dR_sum_nn
plot_map_pair(
    dR_nn_residual.sel(month=month, year=year) / SECONDS_PER_DAY,
    r'$\Delta R_{res}$' + f', {year}-{month:02d}',
    'dR_res',
)

## Kernel Responses

In [ ]:
K_a = xr.open_dataset('data/ERA5_kernels/ERA5_kernel_alb_TOA.nc')
da = anomaly.fal.interp(latitude=K_a.latitude, longitude=K_a.longitude).compute()

dR_a_k = K_a.TOA_all * da
dR_a_clr = K_a.TOA_clr * da

for field, suffix, title_suffix in [(dR_a_k, 'a', 'a'), (dR_a_clr, 'a_clr', r'a,clr')]:
    plot_map_pair(
        field.sel(month=month, year=year) / SECONDS_PER_DAY,
        r'$\Delta R_{' + title_suffix + '}$' + f', {year}-{month:02d}',
        f'k_dR_{suffix}',
    )

In [ ]:
K_q = xr.open_dataset('data/ERA5_kernels/layer_specified_ta_wv_kernel/ERA5_kernel_wv_sw_nodp_TOA.nc').fillna(0)
ds_qt = xr.load_dataset('data/era5/era5_plev_qt_monthly_downscaled.nc')
"""
ds_qt = xr.open_mfdataset('data/era5/era5_plev_qt_monthly_*.nc')
#ds_qt = ds_qt.sel(date=slice('2007-01', '2016-12'))
#ds_qt = ds_qt.interp(latitude=K_q.latitude, longitude=K_q.longitude)
#ds_qt.to_netcdf('data/era5/era5_plev_qt_monthly_downscaled.nc')
"""

ds_qt_monthly = to_monthly(ds_qt.copy(deep=True))
ds_qt_anomaly = ds_qt_monthly - ds_qt_monthly.mean('year')

q = ds_qt_monthly.q
T = ds_qt_monthly.t
dq = ds_qt_anomaly.q

Rv = 461.52  # J / kg / K
Lv = 2260000  # J / kg
sp = ds_monthly.sp.interp(latitude=K_q.latitude, longitude=K_q.longitude).compute()

# dT from dq using Clausius-Clapeyron approximation.
dT = (dq / q) * (Rv / Lv) * (T**2)

dR_q_k = integrate_over_pressure_levels(sp, K_q.TOA_all * dT).compute()
dR_q_clr = integrate_over_pressure_levels(sp, K_q.TOA_clr * dT).compute()

for field, suffix, title_suffix in [(dR_q_k, 'q', 'q'), (dR_q_clr, 'q_clr', r'q,clr')]:
    plot_map_pair(
        field.sel(month=month, year=year),
        r'$\Delta R_{' + title_suffix + '}$' + f', {year}-{month:02d}',
        f'k_dR_{suffix}',
    )

In [ ]:
kernel_grid = {'latitude': dR_q_k.latitude, 'longitude': dR_q_k.longitude}
dR_k = (anomaly.tsr.interp(**kernel_grid) / SECONDS_PER_DAY).compute()
dR_a_k_wm2 = (dR_a_k.interp(**kernel_grid) / SECONDS_PER_DAY).compute()
dR_a_clr_wm2 = (dR_a_clr.interp(**kernel_grid) / SECONDS_PER_DAY).compute()

all_sky_residual_k = dR_k - dR_a_k_wm2 - dR_q_k
dR_cld_k = (dR_a_k_wm2 - dR_a_clr_wm2) + (dR_q_k - dR_q_clr) - all_sky_residual_k

plot_map_pair(
    dR_cld_k.sel(month=month, year=year),
    r'$\Delta R_{cld,k}$' + f', {year}-{month:02d}',
    'k_dR_cld',
)

## Neural Network Vs Kernel Timeseries

In [ ]:
series = {
    'dR_a_nn': global_date_series(dR_a_nn, SECONDS_PER_DAY),
    'dR_a_k': global_date_series(dR_a_k, SECONDS_PER_DAY),
    'dR_q_nn': global_date_series(dR_q_nn, SECONDS_PER_DAY),
    'dR_q_k': global_date_series(dR_q_k),
}

for var in ['a', 'q']:
    plot_timeseries_comparison(series, var)